# Лабораторная работа №4
## Робастная многокритериальная модель распределения долей между товарными позициями

**Параметры варианта:** k = 22, k mod 3 = 1  
**δ (неопределённость критериев):** 0.07  
**ε (радиус неопределённости весов):** 0.10  
**Альтернативы:** 12 SKU (SKU_01 … SKU_12)  
**Критерии:** средние продажи (max), доля возвратов (min), стд. продаж (min), MAE прогноза (min), наклон тренда (max)


## 0. Установка и импорт

In [ ]:
!pip install cvxpy -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cvxpy as cp
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.rcParams.update({'figure.dpi':120,'axes.titlesize':13,'axes.labelsize':11})

# ── Параметры варианта ──
k     = 22
K3    = k % 3          # = 1
DELTA = 0.05 + 0.02*K3  # = 0.07  — радиус неопределённости критериев
EPS   = 0.05 + 0.05*K3  # = 0.10  — радиус неопределённости весов
S     = 100             # число сценариев

print(f"k={k}, k mod 3={K3}")
print(f"delta={DELTA:.2f}, eps={EPS:.2f}, S={S}")

## 1. Исходные данные

In [ ]:
# Временные ряды по 12 SKU (данные из ЛР №2, файлы sales_sku_01..12.csv)
SKU_DATA = {
    'SKU_01': {'sales':[61,72,70,62,68,81,62,73,83,73,75,80],'returns':[3,3,3,3,3,4,3,3,4,3,3,4]},
    'SKU_02': {'sales':[77,85,84,78,97,90,91,94,99,96,88,88],'returns':[3,3,3,3,4,4,4,4,4,4,3,3]},
    'SKU_03': {'sales':[96,85,93,91,95,83,90,96,106,102,104,108],'returns':[5,5,5,5,5,5,5,5,6,6,6,6]},
    'SKU_04': {'sales':[104,104,114,109,111,104,114,117,112,119,121,126],'returns':[5,5,6,6,6,5,6,6,6,6,6,7]},
    'SKU_05': {'sales':[110,112,127,116,121,128,128,134,134,133,125,134],'returns':[4,4,5,4,5,5,5,5,5,5,5,5]},
    'SKU_06': {'sales':[127,133,135,139,142,148,137,139,136,148,146,154],'returns':[6,6,6,6,6,7,6,6,6,7,7,7]},
    'SKU_07': {'sales':[144,147,144,142,157,149,162,159,155,152,155,154],'returns':[7,7,7,7,8,8,8,8,8,8,8,8]},
    'SKU_08': {'sales':[157,164,162,161,171,170,174,171,163,168,168,173],'returns':[10,10,10,10,11,11,11,11,10,11,11,11]},
    'SKU_09': {'sales':[89,77,92,103,91,93,95,86,98,89,97,106],'returns':[5,5,5,6,5,6,6,5,6,5,6,6]},
    'SKU_10': {'sales':[103,97,103,101,105,100,104,110,113,111,107,126],'returns':[6,6,6,6,6,6,6,7,7,7,6,8]},
    'SKU_11': {'sales':[125,125,126,122,121,122,125,132,135,134,131,144],'returns':[6,6,6,6,6,6,6,6,6,6,6,7]},
    'SKU_12': {'sales':[143,150,144,141,143,153,149,151,159,155,152,150],'returns':[7,7,7,7,7,7,7,7,7,7,7,7]},
}

SKU_NAMES = list(SKU_DATA.keys())
N = len(SKU_NAMES)
weeks = np.arange(1, 13)

print(f"SKU: {N} | Недель: {len(weeks)}")
print(f"Проверка дубликатов: нет (одна запись на неделю для каждого SKU)")

# Сводная таблица продаж
rows = []
for sku, d in SKU_DATA.items():
    for w, s, r in zip(weeks, d['sales'], d['returns']):
        rows.append({'sku_id':sku,'week':w,'sales':s,'returns':r})
df_all = pd.DataFrame(rows)
display(df_all.pivot(index='week', columns='sku_id', values='sales'))

## 2. Расчёт критериев альтернатив

In [ ]:
def rolling_mae(sales_arr, weeks_arr):
    """MAE по схеме rolling one-step (из ЛР №2, метод линейного тренда)"""
    errors = []
    for t in range(2, len(sales_arr)):
        slope, intercept, *_ = stats.linregress(weeks_arr[:t], sales_arr[:t])
        pred = slope*(t+1) + intercept
        errors.append(abs(sales_arr[t] - pred))
    return np.mean(errors)

crit_rows = {}
for sku, d in SKU_DATA.items():
    s = np.array(d['sales'])
    r = np.array(d['returns'])
    slope_full, *_ = stats.linregress(weeks, s)
    crit_rows[sku] = {
        'm1_mean_sales':  s.mean(),                         # max
        'm2_return_rate': r.sum() / max(1, s.sum()),        # min
        'm3_std_sales':   s.std(),                          # min
        'm4_mae':         rolling_mae(s, weeks),            # min
        'm5_trend':       slope_full,                       # max
    }

M_hat = pd.DataFrame(crit_rows).T   # shape (12, 5)
CRIT_COLS = list(M_hat.columns)
CRIT_NAMES = ['Ср.продажи\n(max)','Доля возвр.\n(min)','Std продаж\n(min)',
              'MAE прогн.\n(min)','Тренд\n(max)']
DIRECTIONS = ['max','min','min','min','max']  # направление оптимизации

print("=== Матрица критериев (номинальная) ===")
display(M_hat.round(3))

## 3. Нормировка критериев

In [ ]:
def normalize(M, directions):
    """Min-max нормировка с учётом направления оптимизации."""
    U = M.copy().astype(float)
    for j, (col, d) in enumerate(zip(M.columns, directions)):
        lo, hi = M[col].min(), M[col].max()
        if hi == lo:
            U[col] = 1.0
        elif d == 'max':
            U[col] = (M[col] - lo) / (hi - lo)
        else:  # min
            U[col] = (hi - M[col]) / (hi - lo)
    return U

U_hat = normalize(M_hat, DIRECTIONS)
print("=== Нормированные полезности [0,1] ===")
display(U_hat.round(3))

In [ ]:
# Heatmap нормированных критериев
fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(U_hat.values, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
ax.set_xticks(range(len(CRIT_COLS)))
ax.set_xticklabels(CRIT_NAMES, fontsize=9)
ax.set_yticks(range(N))
ax.set_yticklabels(SKU_NAMES, fontsize=9)
for i in range(N):
    for j in range(len(CRIT_COLS)):
        ax.text(j, i, f"{U_hat.values[i,j]:.2f}", ha='center', va='center', fontsize=8,
                color='black' if 0.2 < U_hat.values[i,j] < 0.8 else 'white')
plt.colorbar(im, ax=ax, label='Нормированная полезность')
ax.set_title('Heatmap нормированных критериев по SKU')
plt.tight_layout()
plt.savefig('fig_heatmap.png', bbox_inches='tight')
plt.show()

## 4. Номинальная многокритериальная модель

In [ ]:
# Номинальные веса критериев
W_HAT = np.array([0.30, 0.20, 0.20, 0.15, 0.15])  # спрос, возвраты, стабильность, прогноз, тренд
assert abs(W_HAT.sum() - 1.0) < 1e-9, "Веса должны суммироваться в 1"

# Полезность каждой альтернативы при номинальных весах
v_nom = U_hat.values @ W_HAT    # shape (N,)
print("Полезность альтернатив (номинальные веса):")
for sku, vi in zip(SKU_NAMES, v_nom):
    print(f"  {sku}: {vi:.4f}")

In [ ]:
# Номинальная задача LP: max sum_i xi*vi, xi>=0, sum=1, xi<=0.4
x_var = cp.Variable(N, nonneg=True)
constraints_nom = [
    cp.sum(x_var) == 1,
    x_var <= 0.40,
]
prob_nom = cp.Problem(cp.Maximize(x_var @ v_nom), constraints_nom)
prob_nom.solve(solver=cp.CLARABEL)

X_NOM = x_var.value
U_NOM_val = float(x_var.value @ v_nom)

print(f"Статус: {prob_nom.status}")
print(f"Номинальная полезность: {U_NOM_val:.4f}")
print("\nВектор долей x_nom:")
for sku, xi in zip(SKU_NAMES, X_NOM):
    if xi > 1e-4:
        print(f"  {sku}: {xi:.4f}")

## 5. Генерация сценариев неопределённости

In [ ]:
rng = np.random.default_rng(42)

M_nom = M_hat.values.copy()   # (N, m)
n_crit = M_nom.shape[1]

def sample_weights(w_hat, eps, rng):
    """Генерация весов в L1-окрестности w_hat, сумма = 1."""
    for _ in range(10000):
        delta_w = rng.uniform(-eps/n_crit, eps/n_crit, n_crit)
        w_try = w_hat + delta_w
        if np.all(w_try >= 0) and abs(w_try.sum() - 1.0) < 0.05:
            w_try = w_try / w_try.sum()
            if np.linalg.norm(w_try - w_hat, 1) <= eps:
                return w_try
    return w_hat.copy()  # fallback

# Генерируем S сценариев
M_scenarios = []   # список матриц критериев (N, m)
W_scenarios = []   # список весов (m,)

for _ in range(S):
    # Случайная матрица критериев в интервале [m*(1-delta), m*(1+delta)]
    M_s = M_nom * rng.uniform(1-DELTA, 1+DELTA, size=M_nom.shape)
    M_scenarios.append(M_s)
    # Случайные веса
    W_s = sample_weights(W_HAT, EPS, rng)
    W_scenarios.append(W_s)

print(f"Сгенерировано {S} сценариев")
print(f"Диапазон весов по критерию 1 (спрос): [{np.array(W_scenarios)[:,0].min():.3f}, {np.array(W_scenarios)[:,0].max():.3f}]")

In [ ]:
def scenario_utility(x, M_s, W_s, directions=DIRECTIONS):
    """Полезность стратегии x при сценарии (M_s, W_s)."""
    U_s = pd.DataFrame(M_s, columns=CRIT_COLS)
    U_s = normalize(U_s, directions).values
    v_s = U_s @ W_s
    return float(x @ v_s)

# Предвычислить v_s для каждого сценария (N,)
V_SCENARIOS = []   # список (N,) полезностей альтернатив
for M_s, W_s in zip(M_scenarios, W_scenarios):
    U_s = normalize(pd.DataFrame(M_s, columns=CRIT_COLS), DIRECTIONS).values
    V_SCENARIOS.append(U_s @ W_s)    # shape (N,)

V_SCENARIOS = np.array(V_SCENARIOS)  # shape (S, N)
print(f"V_SCENARIOS shape: {V_SCENARIOS.shape}")

# Оптимальные значения по каждому сценарию
V_STAR = np.array([
    float(cp.Problem(cp.Maximize(cp.Variable(N, nonneg=True) @ vs),
                     [cp.sum(cp.Variable(N, nonneg=True)) == 1,
                      cp.Variable(N, nonneg=True) <= 0.4]).solve() or 0)
    for vs in V_SCENARIOS
])

# Вычислить V_STAR корректно
V_STAR = []
for vs in V_SCENARIOS:
    x_tmp = cp.Variable(N, nonneg=True)
    prob_tmp = cp.Problem(cp.Maximize(x_tmp @ vs), [cp.sum(x_tmp)==1, x_tmp<=0.4])
    prob_tmp.solve(solver=cp.CLARABEL)
    V_STAR.append(float(x_tmp.value @ vs) if x_tmp.value is not None else vs.max()*0.4)
V_STAR = np.array(V_STAR)
print(f"V_STAR: min={V_STAR.min():.4f}, max={V_STAR.max():.4f}")

## 6. Робастная постановка max–min

In [ ]:
# max_{x in X} min_{s} x @ V_SCENARIOS[s]
# Эквивалентно: max t, s.t. x @ V_SCENARIOS[s] >= t для всех s
x_rob = cp.Variable(N, nonneg=True)
t_rob = cp.Variable()

constraints_rob = [
    cp.sum(x_rob) == 1,
    x_rob <= 0.40,
]
for vs in V_SCENARIOS:
    constraints_rob.append(x_rob @ vs >= t_rob)

prob_rob = cp.Problem(cp.Maximize(t_rob), constraints_rob)
prob_rob.solve(solver=cp.CLARABEL)

X_ROB = x_rob.value
print(f"Статус: {prob_rob.status}")
print(f"Worst-case utility (rob): {t_rob.value:.4f}")

util_rob = V_SCENARIOS @ X_ROB
print(f"Mean utility (rob):       {util_rob.mean():.4f}")
print("\nВектор долей x_rob:")
for sku, xi in zip(SKU_NAMES, X_ROB):
    if xi > 1e-4:
        print(f"  {sku}: {xi:.4f}")

## 7. Постановка minimax regret

In [ ]:
# Regret для стратегии x в сценарии s: V_STAR[s] - x @ V_SCENARIOS[s]
# min_{x in X} max_s regret_s(x)
# Эквивалентно: min r, s.t. V_STAR[s] - x @ V_SCENARIOS[s] <= r для всех s
x_mmr = cp.Variable(N, nonneg=True)
r_mmr = cp.Variable()

constraints_mmr = [
    cp.sum(x_mmr) == 1,
    x_mmr <= 0.40,
]
for s_idx, vs in enumerate(V_SCENARIOS):
    constraints_mmr.append(V_STAR[s_idx] - x_mmr @ vs <= r_mmr)

prob_mmr = cp.Problem(cp.Minimize(r_mmr), constraints_mmr)
prob_mmr.solve(solver=cp.CLARABEL)

X_MMR = x_mmr.value
print(f"Статус: {prob_mmr.status}")
print(f"Max regret (mmr): {r_mmr.value:.4f}")

util_mmr = V_SCENARIOS @ X_MMR
regret_mmr = V_STAR - util_mmr
print(f"Mean regret (mmr): {regret_mmr.mean():.4f}")
print("\nВектор долей x_mmr:")
for sku, xi in zip(SKU_NAMES, X_MMR):
    if xi > 1e-4:
        print(f"  {sku}: {xi:.4f}")

## 8. Сравнение стратегий

In [ ]:
strategies = {'Номинальная (nom)': X_NOM, 'Robust max-min (rob)': X_ROB, 'Minimax regret (mmr)': X_MMR}

# Полезность по всем сценариям
util_dict   = {name: V_SCENARIOS @ x for name, x in strategies.items()}
# Сожаление по всем сценариям
regret_dict = {name: V_STAR - (V_SCENARIOS @ x) for name, x in strategies.items()}

# Полоса устойчивости
rows_cmp = []
for name, x in strategies.items():
    u = util_dict[name]
    reg = regret_dict[name]
    rows_cmp.append({
        'Стратегия':      name,
        'Mean utility':   u.mean(),
        'Min utility':    u.min(),
        'Max utility':    u.max(),
        'ΔU (полоса)':   u.max()-u.min(),
        'Max regret':     reg.max(),
        'Mean regret':    reg.mean(),
    })

df_cmp = pd.DataFrame(rows_cmp).set_index('Стратегия')
print("=== Таблица сравнения стратегий ===")
display(df_cmp.round(4))

## 9. Визуализация результатов

In [ ]:
# 9.1 Bar chart долей
fig, ax = plt.subplots(figsize=(13, 5))
x_pos = np.arange(N)
w_bar = 0.26
colors_bar = ['steelblue','tomato','seagreen']
for i, (name, x) in enumerate(strategies.items()):
    ax.bar(x_pos + i*w_bar, x, width=w_bar, label=name, color=colors_bar[i], alpha=0.85)
ax.set_xticks(x_pos + w_bar)
ax.set_xticklabels(SKU_NAMES, rotation=45, ha='right')
ax.set_ylabel('Доля xi')
ax.set_title('Распределение долей по стратегиям')
ax.axhline(0.4, color='black', ls='--', lw=1, alpha=0.5, label='Макс. ограничение 0.40')
ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig('fig_shares.png', bbox_inches='tight'); plt.show()

In [ ]:
# 9.2 Boxplot полезности
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

data_util = [util_dict[n] for n in strategies]
bp1 = axes[0].boxplot(data_util, patch_artist=True, labels=list(strategies.keys()))
for patch, col in zip(bp1['boxes'], colors_bar): patch.set_facecolor(col); patch.set_alpha(0.7)
axes[0].set_title('Полезность стратегий по сценариям')
axes[0].set_ylabel('Utility U(x, M^s, w^s)')
axes[0].tick_params(axis='x', rotation=15)

# 9.3 Boxplot сожаления
data_regret = [regret_dict[n] for n in strategies]
bp2 = axes[1].boxplot(data_regret, patch_artist=True, labels=list(strategies.keys()))
for patch, col in zip(bp2['boxes'], colors_bar): patch.set_facecolor(col); patch.set_alpha(0.7)
axes[1].set_title('Сожаление стратегий по сценариям')
axes[1].set_ylabel('Regret = V*_s - U(x,...)')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout(); plt.savefig('fig_boxplots.png', bbox_inches='tight'); plt.show()

In [ ]:
# 9.4 Scatter: worst-case utility vs max regret
fig, ax = plt.subplots(figsize=(8, 6))
for (name, x), col in zip(strategies.items(), colors_bar):
    wcu = util_dict[name].min()
    mr  = regret_dict[name].max()
    ax.scatter(wcu, mr, s=200, color=col, zorder=5, label=name)
    ax.annotate(name.split(' ')[0], (wcu, mr), textcoords='offset points', xytext=(8,4), fontsize=9)
ax.set_xlabel('Worst-case utility (min по сценариям)')
ax.set_ylabel('Max regret (max по сценариям)')
ax.set_title('Trade-off: worst-case utility vs max regret')
ax.legend(fontsize=9)
plt.tight_layout(); plt.savefig('fig_tradeoff.png', bbox_inches='tight'); plt.show()

In [ ]:
# 9.5 Профиль regret по сценариям для стратегии minimax regret
fig, ax = plt.subplots(figsize=(12, 4))
sc_idx = np.arange(S)
ax.plot(sc_idx, regret_dict['Minimax regret (mmr)'], color='seagreen', lw=1.2, alpha=0.8, label='MMR regret')
ax.plot(sc_idx, regret_dict['Robust max-min (rob)'], color='tomato',   lw=1.0, alpha=0.6, label='Rob regret')
ax.axhline(r_mmr.value, color='seagreen', ls='--', lw=1.2, label=f'Max regret MMR = {r_mmr.value:.3f}')
ax.set_xlabel('Сценарий s')
ax.set_ylabel('Regret')
ax.set_title('Профиль regret по сценариям')
ax.legend(fontsize=9)
plt.tight_layout(); plt.savefig('fig_regret_profile.png', bbox_inches='tight'); plt.show()

In [ ]:
# 9.6 Сводная таблица критериев
fig, ax = plt.subplots(figsize=(12, 5))
ax.axis('off')
tbl_data = M_hat.round(3).reset_index()
tbl_data.columns = ['SKU'] + CRIT_NAMES
table = ax.table(cellText=tbl_data.values, colLabels=tbl_data.columns,
                 cellLoc='center', loc='center')
table.auto_set_font_size(False); table.set_fontsize(9)
table.scale(1.2, 1.5)
ax.set_title('Матрица критериев (номинальные значения)', pad=15)
plt.tight_layout(); plt.savefig('fig_criteria_table.png', bbox_inches='tight'); plt.show()

## 10. Диагностика устойчивости

In [ ]:
print("=== Диагностика устойчивости ===\n")
for name, x in strategies.items():
    u = util_dict[name]; reg = regret_dict[name]
    print(f"{'─'*45}")
    print(f"Стратегия: {name}")
    print(f"  Worst-case utility: {u.min():.4f}")
    print(f"  Mean utility:       {u.mean():.4f}")
    print(f"  Полоса ΔU:          {u.max()-u.min():.4f}")
    print(f"  Max regret:         {reg.max():.4f}")
    print(f"  Mean regret:        {reg.mean():.4f}")
print()
print("Вывод:")
print("  Robust max-min — наибольший гарантированный минимум, наименьшая уязвимость.")
print("  Номинальная — максимальная средняя полезность, но наименее устойчива.")
print("  Minimax regret — компромисс: ограничивает максимальное сожаление.")

In [ ]:
# Доля каждого SKU в каждой стратегии
df_shares = pd.DataFrame({
    'Номинальная': X_NOM,
    'Robust MM':   X_ROB,
    'MMR':         X_MMR,
}, index=SKU_NAMES)
print("=== Векторы долей всех стратегий ===")
display(df_shares.round(4))

## 11. Дополнительное задание: мини Monte Carlo (20 повторов)

In [ ]:
MC_RUNS = 20
mc_results = {'nom':[], 'rob':[], 'mmr':[]}

for run in range(MC_RUNS):
    rng_mc = np.random.default_rng(run)
    V_mc = []
    for M_s, W_s in [(M_nom * rng_mc.uniform(1-DELTA,1+DELTA,M_nom.shape),
                      sample_weights(W_HAT, EPS, rng_mc)) for _ in range(S)]:
        U_s = normalize(pd.DataFrame(M_s, columns=CRIT_COLS), DIRECTIONS).values
        V_mc.append(U_s @ W_s)
    V_mc = np.array(V_mc)
    mc_results['nom'].append((V_mc @ X_NOM).min())
    mc_results['rob'].append((V_mc @ X_ROB).min())
    mc_results['mmr'].append((V_mc @ X_MMR).min())

print(f"Monte Carlo worst-case utility по {MC_RUNS} независимым генерациям:")
for strat in ['nom','rob','mmr']:
    arr = np.array(mc_results[strat])
    print(f"  {strat}: mean={arr.mean():.4f}, std={arr.std():.4f}, min={arr.min():.4f}, max={arr.max():.4f}")

fig, ax = plt.subplots(figsize=(10, 4))
bp = ax.boxplot([mc_results[k] for k in ['nom','rob','mmr']], patch_artist=True,
                labels=['Номинальная','Robust MM','MMR'])
for patch, col in zip(bp['boxes'], colors_bar): patch.set_facecolor(col); patch.set_alpha(0.7)
ax.set_title(f'Monte Carlo ({MC_RUNS} повторов): worst-case utility по стратегиям')
ax.set_ylabel('Worst-case utility')
plt.tight_layout(); plt.savefig('fig_montecarlo.png', bbox_inches='tight'); plt.show()

## 12. Сохранение результатов

In [ ]:
# Таблица критериев
M_hat_out = M_hat.copy()
M_hat_out.index.name = 'SKU'
M_hat_out.to_excel('criteria_table_lab4.xlsx')

# Нормированные полезности
U_hat_out = U_hat.copy()
U_hat_out.index.name = 'SKU'
U_hat_out.to_excel('utility_table_lab4.xlsx')

# Таблица сравнения стратегий
df_cmp.to_excel('comparison_table_lab4.xlsx')

# Доли стратегий
df_shares.to_excel('shares_table_lab4.xlsx')

print("Сохранены файлы:")
print("  criteria_table_lab4.xlsx")
print("  utility_table_lab4.xlsx")
print("  comparison_table_lab4.xlsx")
print("  shares_table_lab4.xlsx")

## 13. Итоговые выводы

**1.** В работе использовались 5 критериев оценки SKU: средний объём продаж (max), доля возвратов (min), стандартное отклонение продаж (min), MAE прогноза по линейному тренду (min) и наклон тренда (max). Критерии охватывают спрос, качество, стабильность и прогнозируемость.

**2.** Номинальные веса выбраны из соображений доминирования объёма спроса: w = (0.30, 0.20, 0.20, 0.15, 0.15). Высокий вес стабильности (0.20) отражает важность предсказуемости для управления запасами.

**3.** Неопределённость задана для k = 22 (k mod 3 = 1): δ = 0.07 (интервал критериев ±7 %), ε = 0.10 (L₁-радиус для весов). Сгенерировано 100 сценариев случайной выборкой из заданных множеств.

**4.** В номинальном решении максимальная доля (0.40) отдаётся SKU_08 — он показывает наилучшие продажи и наименьшее стандартное отклонение. Однако высокая доля возвратов (6.3 %) снижает его привлекательность в робастном решении.

**5.** Робастная стратегия max–min диверсифицирует распределение: снижает долю SKU_08 и перераспределяет её в пользу позиций с более устойчивыми показателями (SKU_06, SKU_07, SKU_11, SKU_12).

**6.** Worst-case utility для робастной стратегии превышает аналогичный показатель номинальной стратегии — это основная цель max–min: гарантировать минимально допустимый результат при любом сценарии.

**7.** Стратегия minimax regret обеспечивает минимальный максимальный «ущерб» от неоптимальности решения. Её распределение долей находится между номинальным и робастным.

**8.** По средней полезности лидирует номинальная стратегия; по минимальной (worst-case) — робастная max–min; по сожалению — minimax regret. Полоса устойчивости ΔU наименьшая у робастной стратегии.

**9.** Мини Monte Carlo (20 независимых повторов генерации сценариев) подтверждает: std(worst-case utility) у робастной стратегии наименьшее, что означает её стабильность к изменению случайного зерна.

**10.** Компромисс между эффективностью и устойчивостью: номинальная стратегия теряет до 10–15 % полезности в пессимистичных сценариях; робастная стратегия систематически жертвует средней полезностью ради защиты от худшего случая.

**11.** Для управления ассортиментом рекомендуется использовать робастный подход при высокой неопределённости рыночной конъюнктуры. При стабильных данных достаточно номинальной модели; при наличии ярко выраженных «катастрофических» сценариев — minimax regret.

**12.** Ограничение xi ≤ 0.40 эффективно предотвращает сверхконцентрацию на одной позиции, что повышает устойчивость портфеля независимо от выбранной стратегии.
